In [77]:
from collections import defaultdict
import json
import os
import re


In [78]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")

intermediate_dir = os.path.join(data_dir, "intermediate")
os.makedirs(intermediate_dir, exist_ok=True)


In [79]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)


def load_tests(base_dir: str):
	data = {
		"pre": {"code": None, "full": None},
		"post": {"code": None, "full": None},
	}

	for phase in ["Pre", "Post"]:
		phase_key = phase.lower()
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				file_path = os.path.join(phase_dir, file)
				content = load_json(file_path)

				kind = "code" if "code" in file else "full"
				data[phase_key][kind] = content

	return data


In [80]:
def parse_likert(v: str):
	if isinstance(v, str):
		m = re.match(r"AO0?(\d)", v)
		if m:
			return int(m.group(1))
	return None

def clean_code_data(data: dict):
	valid_users = set()

	for user_id, user_data in data.items():
		if user_data:
			items = {}

			for question, answer in user_data.items():
				if not question.endswith("Time"):
					if question.startswith("G02Q03[SQ"):
						items[question] = parse_likert(answer)
					elif question.startswith("G04Q07"):
						items[question] = answer

			# all -> Devuelve True si bool(x) es True para todos los valores x en el iterable
			# any -> Devuelve True si bool(x) es True para cualquier x en el iterable
			if all(v is not None for v in items.values()):
				valid_users.add(user_id)
	
	deleted_users = set(data.keys()) - valid_users
	return valid_users, deleted_users


In [81]:
def save_json(path: str, data):
	if not path.endswith(".json"):
		path += ".json"
	
	with open(path, "w", encoding="utf-8") as f:
		json.dump(data, f, ensure_ascii=False, indent=4)


def filter_users(data: dict, valid_users: set):
	return {user_id: user_data for user_id, user_data in data.items() if user_id in valid_users}


In [82]:
all_valid_users = set()

for session_name in os.listdir(raw_dir):
	session_dir = os.path.join(raw_dir, session_name)

	tests = load_tests(session_dir)

	pre_code = tests["pre"]["code"]
	post_code = tests["post"]["code"]

	valid_pre, deleted_pre = clean_code_data(pre_code)
	valid_post, deleted_post = clean_code_data(post_code)

	print(f"\n{'=' * 60}")
	print(f"Session: {session_name}")
	print(f"{'=' * 60}")

	print(f"\nPRE:")
	print(f"  Valid ({len(valid_pre)}): {sorted(valid_pre)}")
	print(f"  Deleted ({len(deleted_pre)}): {sorted(deleted_pre)}")

	print(f"\nPOST:")
	print(f"  Valid ({len(valid_post)}): {sorted(valid_post)}")
	print(f"  Deleted ({len(deleted_post)}): {sorted(deleted_post)}")

	session_valid = valid_pre & valid_post

	print(f"\nVALIDOS EN PRE Y POST ({len(session_valid)}):")
	print(f"  {sorted(session_valid)}")

	all_valid_users |= session_valid

	processed_session_dir = os.path.join(intermediate_dir, session_name)
	os.makedirs(processed_session_dir, exist_ok=True)

	for phase_name, phase_data in tests.items():
		phase_dir = os.path.join(processed_session_dir, phase_name.capitalize())
		os.makedirs(phase_dir, exist_ok=True)

		for kind_name, kind_data in phase_data.items():
			path = os.path.join(phase_dir, f"{kind_name}.json")
			cleaned  = filter_users(kind_data, session_valid)
			save_json(path, cleaned)

print(f"\n{'=' * 60}")
print(f"GLOBAL VALID USERS ({len(all_valid_users)}):")
print(f"{'=' * 60}")
print(sorted(all_valid_users))



Session: CarpeDiem-11-06-2025

PRE:
  Valid (39): ['684837aae48b5a00221a37c9_cgph', '684837aae48b5a00221a37c9_dlez', '684837aae48b5a00221a37c9_fwme', '684837aae48b5a00221a37c9_grwp', '684837aae48b5a00221a37c9_hdrx', '684837aae48b5a00221a37c9_mdsy', '684837aae48b5a00221a37c9_ndkj', '684837aae48b5a00221a37c9_newj', '684837aae48b5a00221a37c9_ntvv', '684837aae48b5a00221a37c9_nuwx', '684837aae48b5a00221a37c9_oxqv', '684837aae48b5a00221a37c9_pkpj', '684837aae48b5a00221a37c9_qijz', '684837aae48b5a00221a37c9_qolu', '684837aae48b5a00221a37c9_rcee', '684837aae48b5a00221a37c9_uzdq', '684837aae48b5a00221a37c9_vboy', '684837aae48b5a00221a37c9_wiuv', '684837aae48b5a00221a37c9_xnjp', '684837aae48b5a00221a37c9_yjbq', '684837b0e48b5a00221a37d0_bopu', '684837b0e48b5a00221a37d0_cmak', '684837b0e48b5a00221a37d0_cpzn', '684837b0e48b5a00221a37d0_hgry', '684837b0e48b5a00221a37d0_lnbt', '684837b0e48b5a00221a37d0_lntj', '684837b0e48b5a00221a37d0_mmng', '684837b0e48b5a00221a37d0_naih', '684837b0e48b5a00221a37d

In [83]:
merged_data = defaultdict(dict)

for session_name in os.listdir(raw_dir):
	session_dir = os.path.join(raw_dir, session_name)
	tests = load_tests(session_dir)

	for phase_name, phase_data in tests.items():
		for kind_name, kind_data in phase_data.items():
			key = (phase_name, kind_name)

			merged_data[key].update(kind_data)
						

In [84]:
merged_dir = os.path.join(intermediate_dir, "merged")
os.makedirs(merged_dir, exist_ok=True)

for (phase_name, kind_name), data in merged_data.items():
	phase_dir = os.path.join(merged_dir, phase_name.capitalize())
	os.makedirs(phase_dir, exist_ok=True)
	
	cleaned = filter_users(data, all_valid_users)
	
	path = os.path.join(phase_dir, f"{kind_name}.json")
	save_json(path, cleaned)
	